<a href="https://colab.research.google.com/github/Sakshiiikashyap/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This notebook turns the validated Week-5/Week-6 refresh/content-opportunity model into a practical content action playbook.

The output is **decision-support for human review**, not an automatic publishing or refresh system. The queue is ranked by measured model score plus transparent reason codes, with explicit limits, review rules, monitoring triggers, and paper-ready exports.


## 1. Ranked actions + reason codes

The queue prioritises pages that are more likely to be worth review, while keeping the recommendation explainable.

**Action logic**
- `REFRESH_REVIEW_CTR`: low CTR is present; review title/snippet/search intent before deciding whether to refresh.
- `REFRESH_REVIEW_STALENESS`: the page has been untouched for a long time; review factual freshness and SERP alignment.
- `REFRESH_REVIEW_CTR_AND_STALENESS`: both signals are present, so the page gets higher review priority.
- `MONITOR`: no strong action signal; keep the page under observation rather than forcing a change.

**Archetype → action mapping**
- **Mature + low CTR:** refresh candidate; inspect title, snippet, intent match, and SERP competitiveness.
- **Mature + acceptable CTR:** monitor unless another business/content signal justifies review.
- **Recently updated + low CTR:** review CTR/search intent first; do not assume the page needs a full content rewrite.
- **Recently updated + acceptable CTR:** monitor.

The model score ranks the queue; the reason code explains the observable signal behind the recommended review. The final action remains human-reviewed.


In [2]:
import zipfile
from pathlib import Path

ZIP_PATH = Path("/content/flyrank-ml-internship-main.zip")
EXTRACT_PATH = Path("/content/flyrank_project")

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("✅ ZIP extracted!")
print("CSV files found:")

for p in EXTRACT_PATH.rglob("content_refresh_anonymized.csv"):
    print(p)

✅ ZIP extracted!
CSV files found:
/content/flyrank_project/flyrank-ml-internship-main/data/raw/content_refresh_anonymized.csv


In [3]:
# Section 1 check: build a ranked action queue with reason codes.
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
FEATURES = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
]

# Locate or extract the FlyRank data.
csv_name = "content_refresh_anonymized.csv"
candidate_paths = [
    Path.cwd() / "data" / "raw" / csv_name,
    Path.cwd().parent / "data" / "raw" / csv_name,
    Path.cwd().parent.parent / "data" / "raw" / csv_name,
    Path("/content/flyrank_project") / "flyrank-ml-internship-main" / "data" / "raw" / csv_name,
    Path("/content/flyrank_project") / "data" / "raw" / csv_name,
    Path("/content/flyrank-ml-internship-main") / "data" / "raw" / csv_name,
]

DATA_PATH = next((p for p in candidate_paths if p.exists()), None)

if DATA_PATH is None:
    zip_candidates = list(Path("/content").glob("*.zip"))
    if zip_candidates:
        extract_path = Path("/content/flyrank_project")
        extract_path.mkdir(exist_ok=True)
        with zipfile.ZipFile(zip_candidates[0], "r") as z:
            z.extractall(extract_path)
        matches = list(extract_path.rglob(csv_name))
        if matches:
            DATA_PATH = matches[0]

if DATA_PATH is None:
    raise FileNotFoundError("Could not locate content_refresh_anonymized.csv.")

# Repo root = folder containing data/, work/, etc.
PROJECT_ROOT = DATA_PATH
while PROJECT_ROOT.parent != PROJECT_ROOT and not (PROJECT_ROOT / "work").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "work").exists():
    PROJECT_ROOT = DATA_PATH.parents[2]

df = pd.read_csv(DATA_PATH)
df["is_declining_label"] = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)

X = df[FEATURES].replace([np.inf, -np.inf], np.nan)
y = df["is_declining_label"].astype(int)

def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    k = min(k, len(y_true))
    order = np.argsort(-scores, kind="mergesort")[:k]
    return float(y_true[order].mean())

# Honest client-holdout validation, matching the conservative design from Week 6.
clients = df["client_id"].astype(str)
unique_clients = clients.dropna().unique()

if len(unique_clients) >= 5:
    train_clients, test_clients = train_test_split(
        unique_clients, test_size=0.20, random_state=RANDOM_STATE
    )
    train_mask = clients.isin(train_clients)
    test_mask = clients.isin(test_clients)
else:
    train_mask, test_mask = train_test_split(
        np.arange(len(df)),
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y,
    )
    train_mask = pd.Series(False, index=df.index)
    test_mask = pd.Series(False, index=df.index)
    train_mask.iloc[train_mask if isinstance(train_mask, np.ndarray) else []] = True
    test_mask.iloc[test_mask if isinstance(test_mask, np.ndarray) else []] = True

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=25,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )),
])

model.fit(X.loc[train_mask], y.loc[train_mask])
test_scores = model.predict_proba(X.loc[test_mask])[:, 1]
test_p50 = precision_at_k(y.loc[test_mask], test_scores, 50)

# Fit on all available rows for the operational ranking.
model.fit(X, y)
df["model_score"] = model.predict_proba(X)[:, 1]

# Transparent reason codes based only on observable input signals.
low_ctr = df["ctr"].fillna(np.inf) < 0.5
stale = df["days_since_last_update"].fillna(0) >= 180

df["reason_code"] = np.select(
    [
        low_ctr & stale,
        low_ctr,
        stale,
    ],
    [
        "LOW_CTR_AND_STALENESS",
        "LOW_CTR",
        "STALE_CONTENT",
    ],
    default="NO_STRONG_SIGNAL",
)

df["action"] = np.select(
    [
        df["reason_code"].eq("LOW_CTR_AND_STALENESS"),
        df["reason_code"].eq("LOW_CTR"),
        df["reason_code"].eq("STALE_CONTENT"),
    ],
    [
        "REFRESH_REVIEW_CTR_AND_STALENESS",
        "REFRESH_REVIEW_CTR",
        "REFRESH_REVIEW_STALENESS",
    ],
    default="MONITOR",
)

df["archetype"] = np.select(
    [
        stale & low_ctr,
        stale & ~low_ctr,
        ~stale & low_ctr,
    ],
    [
        "Mature + low CTR",
        "Mature + acceptable CTR",
        "Recently updated + low CTR",
    ],
    default="Recently updated + acceptable CTR",
)

# Rank action-worthy rows first, then model score.
df["action_priority"] = np.where(df["action"].eq("MONITOR"), 0, 1)
queue = df.sort_values(
    ["action_priority", "model_score"],
    ascending=[False, False],
    kind="mergesort",
).copy()

queue.insert(0, "rank", np.arange(1, len(queue) + 1))

queue_cols = [
    "rank",
    "content_id",
    "client_id",
    "model_score",
    "archetype",
    "reason_code",
    "action",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
]
queue = queue[queue_cols]

print(f"Rows: {len(df):,}")
print(f"Honest client-holdout Precision@50: {test_p50:.3f}")
print("\nTop 10 ranked actions:")
display(queue.head(10))


Rows: 30,000
Honest client-holdout Precision@50: 0.500

Top 10 ranked actions:


,rank,content_id,client_id,model_score,archetype,reason_code,action,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count
28154,1,content_4d76cdb3387b,client_3fdba35f04,0.886169,Recently updated + low CTR,LOW_CTR,REFRESH_REVIEW_CTR,131,104,1597,2.7,0.13,1554.0
17547,2,content_f552433bab3c,client_3fdba35f04,0.885324,Recently updated + low CTR,LOW_CTR,REFRESH_REVIEW_CTR,124,104,337,1.8,0.00,1382.0
23015,3,content_1c539e7bac93,client_3fdba35f04,0.881585,Recently updated + low CTR,LOW_CTR,REFRESH_REVIEW_CTR,131,104,618,3.6,0.00,1558.0
26453,4,content_9b6df29f7889,client_3fdba35f04,0.877930,Recently updated + low CTR,LOW_CTR,REFRESH_REVIEW_CTR,139,104,1622,3.1,0.12,1415.0
27324,5,content_04f9c8cfb061,client_3fdba35f04,0.877909,Recently updated + low CTR,LOW_CTR,REFRESH_REVIEW_CTR,131,104,1914,3.5,0.10,1531.0
10431,6,content_d77e888445e9,client_3fdba35f04,0.877614,Recently updated + low CTR,LOW_CTR,REFRESH_REVIEW_CTR,124,104,1225,23.6,0.08,1420.0
5732,7,content_b893db8d7619,client_3fdba35f04,0.877003,Recently updated + low CTR,LOW_CTR,REFRESH_REVIEW_CTR,124,104,818,4.5,0.00,1560.0
27535,8,content_e04eb9549989,client_3fdba35f04,0.876673,Recently updated + low CTR,LOW_CTR,REFRESH_REVIEW_CTR,131,104,3393,3.6,0.09,1408.0
4523,9,content_0361d8df96e6,client_3fdba35f04,0.875661,Recently updated + low CTR,LOW_CTR,REFRESH_REVIEW_CTR,131,104,697,14.7,0.00,1496.0
3719,10,content_9a38a54c532e,client_3fdba35f04,0.872895,Recently updated + low CTR,LOW_CTR,REFRESH_REVIEW_CTR,124,104,688,3.5,0.15,1515.0


## 2. Intended use and limits

**Intended use:** a content strategist/editor can use the ranked queue to decide which pages deserve review first. The score is a prioritisation signal, while the reason code and archetype provide a simple starting point for the human review.

**Limits:** the model was evaluated as directional decision-support. It does not prove that a page will decline, that a refresh will improve performance, or that one recommended action is causally optimal. The queue is also based on the available observable features and may miss important business context, search intent changes, seasonality, technical SEO issues, or recent events.

This is deliberately a **review-assistance workflow, not production automation**.


In [4]:
# Section 2 check: summarise intended use, limits, and current queue coverage.
summary = pd.DataFrame([
    {
        "Metric": "Rows in ranked queue",
        "Value": len(queue),
    },
    {
        "Metric": "Pages with a non-monitor action",
        "Value": int((queue["action"] != "MONITOR").sum()),
    },
    {
        "Metric": "Pages marked for monitoring",
        "Value": int((queue["action"] == "MONITOR").sum()),
    },
    {
        "Metric": "Honest client-holdout Precision@50",
        "Value": round(test_p50, 3),
    },
])
display(summary)


,Metric,Value
0,Rows in ranked queue,30000.0
1,Pages with a non-monitor action,25789.0
2,Pages marked for monitoring,4211.0
3,Honest client-holdout Precision@50,0.5


## 3. Human review + the no-go list

Every non-monitor recommendation requires a human check before action.

**Human review rules**
1. Confirm the page's current search intent and whether the query/topic is still relevant.
2. Check whether the title/snippet accurately represents the page and offers a credible reason for low CTR.
3. For stale pages, verify facts, links, examples, product/service details, and other time-sensitive claims.
4. Check for technical causes of traffic change before rewriting content.
5. Check seasonality, promotions, major SERP changes, and other business context.
6. Record the final decision: refresh, minor edit, monitor, or reject recommendation.

**No-go list — do NOT automate**
- publishing or deleting content;
- changing canonical tags, redirects, indexing directives, or other technical SEO controls;
- making medical, legal, financial, or brand-safety judgements;
- changing claims/facts without source verification;
- deciding that a refresh *caused* an expected traffic improvement;
- bulk rewriting pages solely because the model score is high.

The model can prioritise a review queue; it should not make irreversible or high-stakes content decisions by itself.


In [5]:
# Section 3 check: verify that every actionable row has a reason and an explicit human-review requirement.
actionable = queue[queue["action"] != "MONITOR"].copy()

review_check = {
    "actionable_rows": len(actionable),
    "missing_reason_codes": int(actionable["reason_code"].isna().sum()),
    "missing_actions": int(actionable["action"].isna().sum()),
    "human_review_required": True,
    "no_go_automation_defined": True,
}

print(review_check)
assert review_check["missing_reason_codes"] == 0
assert review_check["missing_actions"] == 0
assert review_check["human_review_required"]
assert review_check["no_go_automation_defined"]
print("HUMAN REVIEW / NO-GO CHECK PASSED")


{'actionable_rows': 25789, 'missing_reason_codes': 0, 'missing_actions': 0, 'human_review_required': True, 'no_go_automation_defined': True}
HUMAN REVIEW / NO-GO CHECK PASSED


## 4. Monitoring / retrain triggers

The playbook should be monitored lightly rather than treated as a static rule.

**Monitoring triggers**
- Track Precision@50 on a refreshed labelled sample when labels become available.
- Track the proportion of queue items receiving the intended action after human review.
- Watch for large shifts in feature distributions, especially CTR, impressions, content age, and days since update.
- Watch for a sustained increase in rejected recommendations or repeated no-go cases.

**Retrain triggers**
- Precision@50 falls materially below the previous validated result on a comparable evaluation sample.
- Feature distributions shift substantially because the content portfolio or measurement process changes.
- The definition or measurement window of the declining label changes.
- Enough new labelled observations accumulate to justify a fresh validation.

A retrain should be followed by the same leakage audit and honest grouped validation used in Week 6 before the new model is trusted for ranking.


In [6]:
# Section 4 check: store monitoring and retrain triggers as structured text.
triggers = pd.DataFrame([
    {"Type": "Monitoring", "Trigger": "Precision@50 declines on a newly labelled comparable sample."},
    {"Type": "Monitoring", "Trigger": "Human reviewers increasingly reject or override recommendations."},
    {"Type": "Monitoring", "Trigger": "Large feature-distribution shift in CTR, impressions, age, or update recency."},
    {"Type": "Retrain", "Trigger": "Label definition or measurement window changes."},
    {"Type": "Retrain", "Trigger": "Enough new labelled data accumulates for a fresh validation."},
    {"Type": "Retrain", "Trigger": "Portfolio/process change makes the current model less representative."},
])
display(triggers)


,Type,Trigger
0,Monitoring,Precision@50 declines on a newly labelled comp...
1,Monitoring,Human reviewers increasingly reject or overrid...
2,Monitoring,"Large feature-distribution shift in CTR, impre..."
3,Retrain,Label definition or measurement window changes.
4,Retrain,Enough new labelled data accumulates for a fre...
5,Retrain,Portfolio/process change makes the current mod...


## 5. Exports for the paper

The notebook writes the exact ranked queue and a small metrics receipt into `work/outputs/`. The queue is regenerated by the notebook and is not treated as a committed data artifact.

A compact metrics JSON is also written so the paper can trace its reported validation number back to this notebook.


In [7]:
# Section 5 check: export the ranked queue and paper-ready metrics receipt.
outputs_dir = PROJECT_ROOT / "work" / "outputs"
figures_dir = PROJECT_ROOT / "work" / "figures"
outputs_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

queue_path = outputs_dir / "w07_ranked_action_queue.csv"
metrics_path = outputs_dir / "w07_metrics.json"

queue.to_csv(queue_path, index=False)

metrics = {
    "notebook": "w07_action_playbook.ipynb",
    "metric": "Precision@50",
    "honest_split": "client-holdout",
    "random_state": RANDOM_STATE,
    "test_precision_at_50": round(float(test_p50), 6),
    "queue_rows": int(len(queue)),
    "actionable_rows": int((queue["action"] != "MONITOR").sum()),
    "monitor_rows": int((queue["action"] == "MONITOR").sum()),
    "reason_codes": sorted(queue["reason_code"].dropna().unique().tolist()),
    "note": "Queue is decision-support for human review; not a causal or production automation output.",
}

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Queue exported: {queue_path}")
print(f"Metrics receipt exported: {metrics_path}")
print("\nAction counts:")
display(queue["action"].value_counts().rename_axis("action").reset_index(name="count"))


Queue exported: /content/flyrank_project/flyrank-ml-internship-main/work/outputs/w07_ranked_action_queue.csv
Metrics receipt exported: /content/flyrank_project/flyrank-ml-internship-main/work/outputs/w07_metrics.json

Action counts:


,action,count
0,REFRESH_REVIEW_CTR,25615
1,MONITOR,4211
2,REFRESH_REVIEW_CTR_AND_STALENESS,135
3,REFRESH_REVIEW_STALENESS,39


## Self-check

- [x] Ranked actions and reason codes are defined and backed by code.
- [x] Intended use and limits are explicit.
- [x] Human-review rules and a no-go automation list are explicit.
- [x] Monitoring and retrain triggers are defined.
- [x] Ranked queue and metrics receipt export to `work/outputs/`.
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all).
- [x] No client names, URLs, or private queries are intentionally included.
- [x] Claims use careful wording: observed, measured, directional, decision-support.
- [ ] Committed to the repo under `work/notebooks/`.
